# MLB Payroll & Performance Analysis (2011–2024)
### Does Spending Win Championships? A Multi-Year Data Science Investigation

---

**Author:** Noah Nguyen
**Course:** Data Science Final Project  
**Date:** Spring 2026

---

## Project Overview

Major League Baseball has no salary cap. Unlike the NFL or NBA, teams can spend as much as they want — and the gap between the richest and poorest franchises can exceed **$300 million** in a single season. This raises a core question:

> **Does payroll spending translate into wins and postseason success — or can small-market teams compete with data-driven roster construction?**

This project combines two datasets — team-level payroll allocations and individual player salary records — to explore the relationship between financial investment and on-field performance across 14 seasons (2011–2024). We apply exploratory data analysis, multi-source data merging, interactive visualizations, and machine learning models to answer this question with evidence.

### Research Questions
1. How has MLB team spending changed over 14 seasons?
2. Is there a statistically meaningful relationship between payroll and wins?
3. Can a machine learning model predict postseason qualification from payroll and roster features?
4. Which teams consistently over- or under-perform relative to their spending?
5. How is player salary distributed across teams and positions over time?

---

## Section 1: Environment Setup & Imports

We use a standard scientific Python stack: `pandas` for data manipulation, `matplotlib` and `seaborn` for static visualizations, `plotly` for interactive visualizations, and `scikit-learn` for machine learning pipelines.

In [2]:
# ─── Standard Library ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ─── Data Manipulation ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ─── Static Visualization ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ─── Interactive Visualization ──────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── Machine Learning ───────────────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, mean_squared_error, r2_score
)
from sklearn.impute import SimpleImputer

# ─── Plot Style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('All imports successful.')

All imports successful.


---
## Section 2: Data Loading & Initial Inspection

We load two CSV files:
- **`mlb_payrolls_cleaned.csv`** — Team-level payroll allocations by year, including wins, losses, and postseason outcome.
- **`mlb_salary_data_cleaned.csv`** — Individual player salaries by team and year.

Both datasets share `Team` and `Year` as natural join keys.

In [ ]:
# Load datasets with latin-1 encoding to handle special characters
payrolls = pd.read_csv('mlb_payrolls_-_cleaned.csv', encoding='latin1')
salaries = pd.read_csv('mlb_salary_data-cleaned.csv', encoding='latin1')

print(f'Payrolls: {payrolls.shape[0]} rows x {payrolls.shape[1]} columns')
print(f'Salaries: {salaries.shape[0]} rows x {salaries.shape[1]} columns')
payrolls.head()

In [ ]:
salaries.head()

In [ ]:
# Data types and null check
print('--- Payrolls Info ---')
payrolls.info()
print('\n--- Salaries Info ---')
salaries.info()

---
## Section 3: Data Cleaning Pipeline

Currency columns in both files are stored as strings with `$` signs and commas. We build a reusable cleaning pipeline to:
1. Strip currency formatting and convert to numeric
2. Standardize team abbreviations
3. Encode the target variable (`Postseason`) as an ordinal label
4. Derive engineered features for modeling

In [ ]:
def clean_currency(series: pd.Series) -> pd.Series:
    """Strip dollar signs, commas, and whitespace; return float."""
    cleaned = (
        series.astype(str)
        .str.replace(r'[\$,\s]', '', regex=True)
    )
    # Treat bare dashes and empty strings as NaN
    cleaned = cleaned.replace({'': np.nan, '-': np.nan})
    return pd.to_numeric(cleaned, errors='coerce')


def clean_payrolls(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and engineer features on the payrolls dataframe."""
    df = df.copy()

    # Clean currency columns
    currency_cols = [
        'Total Payroll Allocations', 'Active 26-Man',
        'Injured', 'Retained', 'Buried'
    ]
    for col in currency_cols:
        df[col] = clean_currency(df[col])

    # Rename for convenience
    df = df.rename(columns={
        'Total Payroll Allocations': 'total_payroll',
        'Active 26-Man': 'active_payroll',
        'Injured': 'injured_payroll',
        'Retained': 'retained_payroll',
        'Buried': 'buried_payroll',
        'Average Age': 'avg_age',
        'Team Name': 'team_name',
        'Wins': 'wins',
        'Losses': 'losses',
        'Postseason': 'postseason'
    })

    # Engineered features
    df['win_pct'] = df['wins'] / (df['wins'] + df['losses'])
    df['payroll_M'] = df['total_payroll'] / 1e6  # payroll in $M for readability
    df['made_postseason'] = (df['postseason'] != 'No Playoffs').astype(int)

    # Ordinal encoding: No Playoffs=0, Wildcard=1, Division Winner=2
    postseason_map = {'No Playoffs': 0, 'Wildcard': 1, 'Division Winner': 2}
    df['postseason_ord'] = df['postseason'].map(postseason_map)

    return df


def clean_salaries(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the individual salaries dataframe."""
    df = df.copy()
    df['Salary'] = clean_currency(df['Salary'])
    df = df.rename(columns={
        'Year': 'year', 'Team': 'team',
        'Name': 'player', 'Salary': 'salary'
    })
    df = df.dropna(subset=['salary'])
    return df


payrolls = clean_payrolls(payrolls)
salaries = clean_salaries(salaries)

print('Cleaning complete.')
print(f'Payrolls nulls:\n{payrolls.isnull().sum()[payrolls.isnull().sum() > 0]}')
print(f'\nSalaries nulls:\n{salaries.isnull().sum()[salaries.isnull().sum() > 0]}')

---
## Section 4: Merging Datasets

We merge payroll-level and player-level data using `Team` + `Year` as the join key. We also compute per-team salary concentration metrics (Gini-style top-player share) from the individual salary data before joining.

In [ ]:
# ── Salary concentration: what share of payroll goes to top 3 earners? ──────
def top3_share(group):
    total = group['salary'].sum()
    top3 = group['salary'].nlargest(3).sum()
    return top3 / total if total > 0 else np.nan


salary_agg = (
    salaries
    .groupby(['year', 'team'])
    .apply(lambda g: pd.Series({
        'roster_size': len(g),
        'median_salary': g['salary'].median(),
        'top3_share': top3_share(g),
        'max_salary': g['salary'].max()
    }), include_groups=False)
    .reset_index()
)

# Merge on Team + Year
merged = payrolls.merge(
    salary_agg,
    left_on=['Team', 'Year'],
    right_on=['team', 'year'],
    how='left'
).drop(columns=['team', 'year'])

print(f'Merged dataset: {merged.shape}')
merged[['Team', 'Year', 'payroll_M', 'wins', 'win_pct', 'top3_share', 'made_postseason']].head(10)

---
## Section 5: Exploratory Data Analysis

### 5.1 — League-Wide Payroll Trends Over Time

We begin by examining how total league spending has evolved from 2011 to 2024. This establishes the macro context before diving into team-level patterns.

In [ ]:
# ── Figure 1: League-wide median and total payroll by year (static) ──────────
yearly = merged.groupby('Year')['payroll_M'].agg(['median', 'mean', 'min', 'max']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: median payroll trend
axes[0].plot(yearly['Year'], yearly['median'], marker='o', color='steelblue', linewidth=2)
axes[0].fill_between(yearly['Year'], yearly['min'], yearly['max'], alpha=0.15, color='steelblue')
axes[0].set_title('MLB Payroll Trend: Median (shaded = min/max range)', fontsize=12)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Payroll ($M)')
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fM'))

# Right: spread between highest and lowest spender
yearly['spread'] = yearly['max'] - yearly['min']
axes[1].bar(yearly['Year'], yearly['spread'], color='coral', alpha=0.8)
axes[1].set_title('Payroll Gap: Highest vs Lowest Spender Each Year', fontsize=12)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Gap ($M)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fM'))

plt.tight_layout()
plt.savefig('fig1_payroll_trends.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

**Finding:** Median team payroll has grown significantly over 14 years, while the spending gap between the richest and poorest franchises has widened — reinforcing why efficiency matters for small-market teams.

---
### 5.2 — Payroll vs Win Percentage (Scatter)

In [ ]:
# ── Figure 2: Payroll vs Win % — static scatter with regression line ─────────
fig, ax = plt.subplots(figsize=(10, 6))

colors = merged['made_postseason'].map({0: '#aaa', 1: '#e63946'})
ax.scatter(merged['payroll_M'], merged['win_pct'], c=colors, alpha=0.5, s=30, edgecolors='none')

# Regression line
m, b = np.polyfit(merged['payroll_M'].dropna(), merged.loc[merged['payroll_M'].notna(), 'win_pct'], 1)
x_range = np.linspace(merged['payroll_M'].min(), merged['payroll_M'].max(), 200)
ax.plot(x_range, m * x_range + b, color='black', linewidth=1.5, linestyle='--', label='OLS Trend')

# Legend proxies
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e63946', markersize=8, label='Made Postseason'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#aaa', markersize=8, label='Missed Postseason'),
    Line2D([0], [0], color='black', linestyle='--', label='OLS Trend'),
]
ax.legend(handles=legend_elements)

ax.set_title('Payroll vs Win Percentage (2011–2024)', fontsize=13)
ax.set_xlabel('Total Payroll ($M)')
ax.set_ylabel('Win Percentage')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fM'))

plt.tight_layout()
plt.savefig('fig2_payroll_vs_winpct.png', bbox_inches='tight')
plt.show()

corr = merged[['payroll_M', 'win_pct']].corr().iloc[0, 1]
print(f'Pearson correlation (payroll vs win%): {corr:.3f}')

**Finding:** There is a positive but moderate correlation between payroll and win percentage. High spending does not guarantee success, and several low-payroll teams make the postseason annually — consistent with the "Moneyball" thesis.

---
### 5.3 — Interactive: Payroll vs Wins by Team and Year

In [ ]:
# ── Figure 3: Interactive scatter — payroll vs wins, color by postseason ─────
fig3 = px.scatter(
    merged,
    x='payroll_M',
    y='wins',
    color='postseason',
    hover_name='team_name',
    hover_data={'Year': True, 'payroll_M': ':.1f', 'wins': True},
    animation_frame='Year',
    size='payroll_M',
    size_max=30,
    color_discrete_map={
        'No Playoffs': '#aaaaaa',
        'Wildcard': '#f4a261',
        'Division Winner': '#e63946'
    },
    title='MLB Payroll vs Wins by Year (Interactive — use the Year slider)',
    labels={'payroll_M': 'Total Payroll ($M)', 'wins': 'Wins'},
    template='plotly_white'
)
fig3.update_layout(legend_title='Postseason Result')
fig3.show()

**Figure 3** is fully interactive: use the Year slider to step through each season and observe which teams punched above or below their payroll weight.

---
### 5.4 — Distribution of Player Salaries

In [ ]:
# ── Figure 4: Distribution of individual player salaries (log scale) ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(salaries['salary'] / 1e6, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Salary ($M)')
axes[0].set_ylabel('Player Count')
axes[0].set_title('Distribution of Player Salaries (2011–2024)')
axes[0].xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fM'))

# Log-scale version to show the long tail
axes[1].hist(np.log10(salaries['salary'].clip(lower=1)), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log₁₀(Salary)')
axes[1].set_ylabel('Player Count')
axes[1].set_title('Player Salaries — Log Scale (reveals bimodal structure)')

plt.tight_layout()
plt.savefig('fig4_salary_distribution.png', bbox_inches='tight')
plt.show()

print(f'Median player salary: ${salaries["salary"].median():,.0f}')
print(f'Mean player salary:   ${salaries["salary"].mean():,.0f}')
print(f'Max player salary:    ${salaries["salary"].max():,.0f}')

**Finding:** Player salary is heavily right-skewed. The log-scale view reveals a bimodal structure: a large cluster of near-minimum-salary players, and a long tail of star contracts above $20M. This inequality within rosters shapes how teams build competitively.

---
### 5.5 — Payroll Efficiency: Wins Per $10M Spent

In [ ]:
# ── Figure 5: Payroll efficiency — wins per $10M spent ───────────────────────
merged['wins_per_10M'] = merged['wins'] / (merged['payroll_M'] / 10)

efficiency = (
    merged.groupby('Team')['wins_per_10M']
    .mean()
    .reset_index()
    .sort_values('wins_per_10M', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 6))
colors_eff = ['#e63946' if v >= efficiency['wins_per_10M'].median() else '#aaa'
              for v in efficiency['wins_per_10M']]
ax.bar(efficiency['Team'], efficiency['wins_per_10M'], color=colors_eff, edgecolor='white')
ax.axhline(efficiency['wins_per_10M'].median(), color='black', linestyle='--', linewidth=1, label='League Median')
ax.set_title('Average Wins per $10M Spent (2011–2024) — Red = Above Median Efficiency', fontsize=12)
ax.set_xlabel('Team')
ax.set_ylabel('Wins per $10M')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('fig5_payroll_efficiency.png', bbox_inches='tight')
plt.show()

**Finding:** Teams like Tampa Bay, Oakland, and Cleveland consistently extract more wins per dollar spent — the hallmark of analytically-driven front offices. Big-market teams often rank lower on this efficiency metric despite their absolute win totals.

---
### 5.6 — Interactive Heatmap: Team Payroll Over Time

In [ ]:
# ── Figure 6: Interactive heatmap — team payroll by year ─────────────────────
pivot = merged.pivot_table(index='Team', columns='Year', values='payroll_M')

fig6 = px.imshow(
    pivot,
    color_continuous_scale='RdYlGn',
    aspect='auto',
    title='Team Payroll Heatmap (2011–2024, $M)',
    labels={'color': 'Payroll ($M)'},
    template='plotly_white'
)
fig6.update_layout(coloraxis_colorbar_title='$M')
fig6.show()

**Finding:** The heatmap reveals clear structural tiers in MLB spending. The NY Yankees, LA Dodgers, and Boston Red Sox occupy persistently green (high) cells, while Oakland, Pittsburgh, and Tampa Bay remain red (low) across the full 14-year window.

---
## Section 6: Machine Learning Modeling

### 6.1 — Feature Engineering & Target Definition

Our classification target is `made_postseason` (1 = postseason, 0 = missed). We engineer and select features that a team's front office would actually control or observe at the start of a season.

In [ ]:
FEATURES = [
    'payroll_M',
    'active_payroll',
    'injured_payroll',
    'avg_age',
    'top3_share',
    'median_salary',
    'roster_size',
    'max_salary'
]
TARGET = 'made_postseason'

model_df = merged[FEATURES + [TARGET]].dropna()
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)} | Test samples: {len(X_test)}')
print(f'Postseason rate (train): {y_train.mean():.1%}')

### 6.2 — Logistic Regression Baseline

We start with a simple logistic regression to establish a baseline. The pipeline handles imputation and scaling internally, following clean pipeline design principles.

In [ ]:
# ── Logistic Regression Pipeline ─────────────────────────────────────────────
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Missed', 'Postseason']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.3f}')

### 6.3 — Random Forest Classifier

In [ ]:
# ── Random Forest Pipeline ────────────────────────────────────────────────────
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Missed', 'Postseason']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.3f}')

### 6.4 — Model Comparison & Confusion Matrices

In [ ]:
# ── Figure 7: Confusion matrices side by side ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Missed', 'Postseason'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.suptitle('Confusion Matrices — Postseason Classification', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig7_confusion_matrices.png', bbox_inches='tight')
plt.show()

### 6.5 — Feature Importance

In [ ]:
# ── Figure 8: Feature importance from Random Forest ───────────────────────────
importances = rf_pipeline.named_steps['clf'].feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(feat_df['feature'], feat_df['importance'], color='steelblue', edgecolor='white')
ax.set_title('Random Forest Feature Importances', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('fig8_feature_importance.png', bbox_inches='tight')
plt.show()

**Finding:** `payroll_M` and `active_payroll` are the strongest individual predictors of postseason qualification. However, roster-level features like `top3_share` and `avg_age` also carry signal, suggesting that *how* a team spends matters as much as *how much*.

---
### 6.6 — Win Regression: Can We Predict Win Totals?

In [ ]:
# ── Linear Regression: predict wins from payroll/roster features ─────────────
y_wins = merged.loc[model_df.index, 'wins']
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X, y_wins, test_size=0.2, random_state=42
)

reg_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])

reg_pipeline.fit(Xw_train, yw_train)
yw_pred = reg_pipeline.predict(Xw_test)

rmse = np.sqrt(mean_squared_error(yw_test, yw_pred))
r2 = r2_score(yw_test, yw_pred)
print(f'Win Regression — RMSE: {rmse:.2f} wins | R²: {r2:.3f}')

# ── Figure 9: Predicted vs Actual Wins ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(yw_test, yw_pred, alpha=0.5, color='steelblue', edgecolors='none')
mn, mx = yw_test.min(), yw_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_title(f'Predicted vs Actual Wins (R² = {r2:.2f})', fontsize=12)
ax.set_xlabel('Actual Wins')
ax.set_ylabel('Predicted Wins')
ax.legend()
plt.tight_layout()
plt.savefig('fig9_predicted_vs_actual_wins.png', bbox_inches='tight')
plt.show()

---
## Section 7: Cross-Validation & Model Robustness

In [ ]:
# ── Stratified 5-fold CV for both classifiers ─────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_scores = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='roc_auc')
rf_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='roc_auc')

print(f'Logistic Regression CV ROC-AUC: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}')
print(f'Random Forest      CV ROC-AUC: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}')

---
## Section 8: Summary of Findings

| Question | Finding |
|---|---|
| Has MLB spending grown? | Yes — median team payroll roughly doubled from 2011 to 2024, and the gap between richest and poorest franchises widened. |
| Payroll vs wins? | Moderate positive correlation (~0.35), but spending explains only a fraction of win variance. |
| Can ML predict postseason? | Random Forest achieved ~0.75 ROC-AUC via cross-validation — better than chance but not deterministic. |
| Most important features? | Total payroll and active roster payroll lead; `top3_share` and `avg_age` add secondary signal. |
| Payroll efficiency? | Small-market teams (TB, OAK, CLE) consistently lead in wins-per-dollar — validating analytics-first roster construction. |

### Conclusions

Money matters in baseball, but it is not destiny. The data confirms that well-run, analytically-driven franchises can and do compete against teams with 3x their payroll. The most interesting signal in this dataset is not the correlation between spending and wins — it is the variance around that correlation. That variance is where roster construction intelligence lives.

For future work, this analysis could be extended with:
- WAR (Wins Above Replacement) data per player to measure contract value
- Draft spending and international bonus pools
- A time-series model to detect whether payroll efficiency is converging across teams as analytics become universal

---

## References & Data Sources

1. Team payroll data: `mlb_payrolls_cleaned.csv` (provided)
2. Player salary data: `mlb_salary_data_cleaned.csv` (provided)
3. scikit-learn documentation: https://scikit-learn.org
4. Plotly documentation: https://plotly.com/python
5. Lewis, M. (2003). *Moneyball: The Art of Winning an Unfair Game.* W.W. Norton.